# Does the c law survive a change of lattice? Hexagonal and triangular ants

## Question
On the square lattice `1/c = 1.36 + 0.70 lead` (c = births per boundary step, lead = opening same-letter run). Is the slope a property of the 2x2 loop, of the lattice
coordination, or of neither? Three hypotheses were registered before any lattice c existed (README): **A** b scales with loop length, **B** b is lattice-independent,
**C** b follows 4/z. Secondary: is the kappa deficit (ants at ~0.8 of the uniform reference) square-specific?

## Assumptions and definitions
Generic octave-window probe on square / hex / triangular lattices (`src/lattice_probe.rs`); boundary step, birth, perimeter as `src/probe.rs`; kappa ratio = kappa / kappa_unif(a, beta)
with the run's own a, beta (k = 20..26). Instrument gates G1-G3 (`cargo test --release --test lattice_probe`) **passed**; G4 (mean-field walker) and G5 (square c law) are checked below.

In [1]:
import hashlib, json, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
R = ROOT / "experiments/archive/2026-10-lattice-c-law/results"
for n in ["ants.csv", "controls.csv", "analysis.json", "analysis.txt", "explore.txt", "run.log"]:
    print(hashlib.sha256((R / n).read_bytes()).hexdigest()[:16], n)
print("git", subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip() or "unresolvable here; see README (instrument adf6677)")
A = json.loads((R / "analysis.json").read_text())

27a19e9fed4225ef ants.csv
4e3d71e477998e66 controls.csv
c9c5c924e719b9f5 analysis.json
337a65fa781e2513 analysis.txt
1eb3e0cf6002d873 explore.txt
ac17e37c160bc507 run.log


git 7339dc9


## Results
### Gates and the c law per stratum (registered fit)

In [2]:
k = A["kappa"]
print("G4 mean-field kappa ratio:", {l: round(k[l]["mf"]["ratio_mean"], 3) for l in k})
for key, f in A["fits"].items():
    print(f"{key:9s} n={f['n']:3d}  1/c = {f['a']:6.2f} + {f['b']:5.2f} lead   b 95% CI [{f['b_ci'][0]:.2f}, {f['b_ci'][1]:.2f}]  R2 {f['r2']:.2f}")
print("retention (kept per lead 1..5):", {key: v["kept"] for key, v in A["retention"].items()})
print("decision rule verdict:", A["verdict"])

G4 mean-field kappa ratio: {'square': 0.994, 'tri': 1.005, 'hex': 1.004}
square/R  n=133  1/c =   1.29 +  0.70 lead   b 95% CI [0.67, 0.72]  R2 0.95
tri/R     n=223  1/c =   1.28 +  0.36 lead   b 95% CI [0.34, 0.37]  R2 0.94
hex/r     n=220  1/c =  -1.18 +  1.98 lead   b 95% CI [0.23, 5.82]  R2 0.01
hex/R     n=174  1/c =   2.80 +  0.88 lead   b 95% CI [0.78, 0.98]  R2 0.69
retention (kept per lead 1..5): {'square/R': [31, 18, 29, 27, 28], 'tri/R': [36, 47, 47, 47, 46], 'hex/r': [38, 41, 47, 47, 47], 'hex/R': [33, 28, 37, 42, 34]}
decision rule verdict: undecided


Square reproduces the census (G5 passes). Triangular b is half the square's, with a tight interval. The hex r-lead fit is corrupted by one rule with 1/c = 390 that passed the frozen filters; the registered decision rule returns **undecided**.
It does **not** show a mechanism for b. The next cell is post hoc and unregistered.

In [3]:
print((R / "explore.txt").read_text())

Reading inline script metadata from `experiments/2026-10-20-lattice-c-law/scripts/explore.py`
== square/R
 lead 1: n=31 median 1/c 2.09  q10/q90 1.51/2.24  max 2.4
 lead 2: n=18 median 1/c 2.62  q10/q90 2.45/2.90  max 2.9
 lead 3: n=29 median 1/c 3.36  q10/q90 3.04/3.61  max 3.8
 lead 4: n=27 median 1/c 4.10  q10/q90 3.89/4.29  max 4.4
 lead 5: n=28 median 1/c 4.82  q10/q90 4.40/4.97  max 5.2
 slope of per-lead medians (lead 1->5)/4: 0.682  OLS on medians: 0.695
== tri/R
 lead 1: n=36 median 1/c 1.60  q10/q90 1.34/1.82  max 2.0
 lead 2: n=47 median 1/c 1.97  q10/q90 1.92/2.04  max 2.1
 lead 3: n=47 median 1/c 2.38  q10/q90 2.32/2.49  max 2.5
 lead 4: n=47 median 1/c 2.65  q10/q90 2.58/2.83  max 3.0
 lead 5: n=46 median 1/c 2.94  q10/q90 2.88/3.27  max 3.3
 slope of per-lead medians (lead 1->5)/4: 0.335  OLS on medians: 0.335
== hex/r
 lead 1: n=38 median 1/c 2.63  q10/q90 2.25/3.72  max 4.7
 lead 2: n=41 median 1/c 2.77  q10/q90 2.40/3.33  max 3.9
 lead 3: n=47 median 1/c 3.04  q10/q90

Per-lead medians give b of about 0.86 (hex R, loop 3), 0.70 (square, loop 4), 0.34 (triangular, loop 6) and 0.27 (hex r, loop 6): **b falls as the loop gets longer**, opposite to H-A.
All three registered hypotheses are disproven as stated: triangular alone is at least 0.33 away from each vector. A line `b ~ 1.5 - 0.2 l` fits but has two parameters for four points chosen after the fact.

### Kappa

In [4]:
for l, v in k.items():
    print(f"{l:6s} ants median ratio {v['ants_median']:.3f} (n={v['ants_n']}, IQR {v['ants_q'][1]:.2f}-{v['ants_q'][2]:.2f})   random walk {v['rw']['ratio_mean']:.2f}   mean-field {v['mf']['ratio_mean']:.3f}")
print("K3 slopes of ln kappa on lead:", {a: round(b, 3) for a, b in A["k3"].items()})

square ants median ratio 0.770 (n=133, IQR 0.62-0.90)   random walk 1.33   mean-field 0.994
tri    ants median ratio 0.744 (n=223, IQR 0.66-0.82)   random walk 1.44   mean-field 1.005
hex    ants median ratio 0.860 (n=394, IQR 0.75-1.01)   random walk 1.28   mean-field 1.004
K3 slopes of ln kappa on lead: {'square/R': -0.007, 'tri/R': -0.079, 'hex/r': -0.059, 'hex/R': -0.075}


The deficit is not square-specific: all three lattices have ant medians of 0.74-0.86 with a mean-field reference at 1.00 and memoryless walkers at 1.3-1.4 (random-walk window k = 16..22). It does not explain the deficit.

## Caveats
- Single deterministic run per rule; survivor selection (e.g. square lead 2 keeps 18 of 36 alive); `alpha_w` is a proxy for the census `blob` class.
- Hex has four letters; letters after the lead are not separated by type. Triangular loops enclose a vertex, so "loop length" may not be the variable that matters.
- Failed registered predictions: L1 (outlier), L2, K4. Deviations: 2^27 steps, no gyration fields, random walk window.

## Next
Register b(l) and a letter-typed hex lead before using a new loop length; derive why b falls with l; check whether the intercept (1.3 square/triangular, ~2.5 hex) tracks coordination.